# Hospital Excess Readmission Classification — Data Cleaning and Structural Validation

This notebook loads the CMS Hospital Readmissions Reduction Program dataset and prepares it for later classification modeling. The purpose of this notebook is to understand the dataset structure, identify missingness patterns, and validate the grain of the data before creating a predictive target.

This project uses hospital-level public CMS HRRP data. Each row is expected to represent a hospital-measure record, not an individual patient readmission.

## Load and Standardize Dataset

The dataset is loaded from the local project data folder. Column names are standardized to lowercase with underscores to make the dataset easier to work with in Python.

In [6]:
#Import path for OS-independent file paths
from pathlib import Path

#Import pandas for data loading and analysis
import pandas as pd

#Relative Path to Dataset for safety
csv_path = Path("../data/raw") / "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv"
df_raw = pd.read_csv(csv_path)

#Standardize column names - convert to lowercase and replace spaces with underscores
df_raw.columns = ( df_raw.columns.
    str.lower(). 
    str.replace(" ", "_")
)

## Initial Data Inspection

Before cleaning, I inspected the dataset shape, column names, data types, missing values, and duplicate rows. This helps identify any structural issues that need to be cleaned before making any changes to the data.

In [7]:
#Inital Inspection of Dataset Structure
df_raw.shape
df_raw.head()
df_raw.info()
df_raw.columns

#Check the number of missing values for each column
df_raw.isna().sum()

#Check for any duplicated rows (0)
df_raw.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 18330 entries, 0 to 18329
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   facility_name               18330 non-null  str    
 1   facility_id                 18330 non-null  int64  
 2   state                       18330 non-null  str    
 3   measure_name                18330 non-null  str    
 4   number_of_discharges        8242 non-null   float64
 5   footnote                    6987 non-null   float64
 6   excess_readmission_ratio    11720 non-null  float64
 7   predicted_readmission_rate  11720 non-null  float64
 8   expected_readmission_rate   11720 non-null  float64
 9   number_of_readmissions      11720 non-null  str    
 10  start_date                  18330 non-null  str    
 11  end_date                    18330 non-null  str    
dtypes: float64(5), int64(1), str(6)
memory usage: 3.0 MB


np.int64(0)

## Missing Value and ERR Reporting Investigation

Several important analytical columns contain missing values, including excess readmission ratio, predicted readmission rate, expected readmission rate, and number of readmissions.

To better understand whether these missing values are random or structurally related to CMS reporting rules, I investigate the relationship between missing ERR values, measure categories, discharge counts, and CMS footnotes.

In [8]:
#Find the correlation between measure categories and missing ERR values
df_raw[df_raw['excess_readmission_ratio'].isna()]['measure_name'].value_counts()

#Find whether discharge counts exist for rows that are missing ERR
df_raw[df_raw['excess_readmission_ratio'].isna()]['number_of_discharges'].value_counts()

#Investigate if there is an association between footnotes and missing ERR values
df_raw[df_raw['excess_readmission_ratio'].isna()]['footnote'].value_counts()


footnote
5.0    3255
1.0    3150
7.0     205
Name: count, dtype: int64

## Create Valid ERR Dataset

Rows missing excess readmission ratio values cannot be used for supervised classification because the target variable depends on ERR availability.

To prepare the analytical dataset, rows missing ERR values are removed while preserving the remaining hospital-measure observations for later modeling analysis.

In [9]:
#Create a filtered analytical dataset containing rows that have valid
# excess readmission ratio values
df_valid_err = df_raw.copy()
df_valid_err = df_valid_err.dropna(subset = ['excess_readmission_ratio'])

#Confirm filtered dataset shape
df_valid_err.shape

(11720, 12)

## Compare Measure Representation After ERR Filtering

After filtering to valid ERR rows, I compare the distribution of measure categories before and after filtering.

This helps identify whether certain HRRP measure categories are disproportionately affected by missing ERR values and whether the filtered dataset remains structurally representative.

In [10]:
# Compare measure distribution after filtering valid ERR rows
df_valid_err['measure_name'].value_counts()

# Compare against original measure distribution before filtering
df_raw['measure_name'].value_counts()

measure_name
READM-30-HIP-KNEE-HRRP    3055
READM-30-CABG-HRRP        3055
READM-30-AMI-HRRP         3055
READM-30-COPD-HRRP        3055
READM-30-PN-HRRP          3055
READM-30-HF-HRRP          3055
Name: count, dtype: int64

## Check Remaining Footnotes and Discharge Volume After Filtering

After filtering the dataset, I inspect the remaining CMS footnotes and compare average discharge counts before and after filtering.

This helps determine whether removing rows with missing ERR values significantly changes the operational characteristics of the dataset.

In [11]:
# Check which footnotes remain after filtering
df_valid_err['footnote'].value_counts()

# Compare average discharge counts before and after filtering
df_raw['number_of_discharges'].mean()
df_valid_err['number_of_discharges'].mean()

np.float64(290.64539007092196)

## Preliminary Target Balance Check

Before creating the final modeling dataset, I perform an initial inspection of the ERR threshold distribution using a classification boundary of ERR > 1.

This provides an early understanding of class balance and confirms how many hospital-measure records fall above and below the excess readmission threshold.

In [12]:
# Preliminary class balance inspection:
# Count rows where ERR indidcates excess readmissions
(df_valid_err['excess_readmission_ratio'] > 1).sum()

# Same check on original dataset
#NaN values evaluate False during comparison)
(df_raw['excess_readmission_ratio'] > 1).sum()

#Count rows where ERR is less than or equal to threshold
(df_valid_err['excess_readmission_ratio'] <= 1).sum()

#Same threshold check on original datset
(df_raw['excess_readmission_ratio'] <= 1).sum()

np.int64(6077)

## Validate Dataset Grain

The dataset grain is validated by checking whether the combination of `facility_id` and `measure_name` uniquely identifies each row.

This confirms that each row represents a unique hospital-measure observation after filtering.

In [13]:
#Confirm filtered dataset shape
df_valid_err.shape
#(11720, 12)

# Check whether facility_id + measure_name uniquely identiifes rows
df_valid_err[['facility_id', 'measure_name']].drop_duplicates().shape[0]
#11720 


11720

## Export Filtered Analytical Dataset

The filtered analytical dataset containing valid excess readmission ratio values is exported as a Parquet file for later target engineering, feature selection, and modeling workflows.

In [14]:
# Save processed dataset for target analysis and modeling notebooks
df_valid_err.to_parquet("../data/processed/valid_err.parquet")